<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/3_12_%EC%9D%B4%EC%BB%A4%EB%A8%B8%EC%8A%A4_%EC%97%90%EC%9D%B4%EC%A0%84%ED%8A%B8_text_to_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain_openai langchain_community gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [69]:
import os
os.environ['OPENAI_API_KEY'] = ''

#SQLite 초기화
DML, DDL 정의

In [3]:
####################################
# 1. 이커머스 관련 테이블 DDL 정의
####################################

ecommerce_schema = """
-- 상품 테이블
CREATE TABLE products (
    product_id VARCHAR(10) PRIMARY KEY,  -- 상품ID (예: P001)
    product_name VARCHAR(100) NOT NULL,  -- 상품명
    warehouse_id VARCHAR(10) NOT NULL,   -- 물류창고ID
    category VARCHAR(50) NOT NULL,       -- 카테고리
    stock_quantity INTEGER NOT NULL,     -- 재고수량
    price INTEGER NOT NULL,              -- 가격
    stocking_date DATE NOT NULL          -- 입고일자
);

-- 고객 테이블
CREATE TABLE customers (
    customer_id VARCHAR(10) PRIMARY KEY, -- 고객ID (예: C001)
    customer_name VARCHAR(50) NOT NULL,  -- 고객명
    birth_date DATE NOT NULL,            -- 생년월일
    gender CHAR(1) NOT NULL,             -- 성별 (M/F)
    phone VARCHAR(20) NOT NULL,          -- 전화번호
    email VARCHAR(100) NOT NULL,         -- 이메일
    join_date DATE NOT NULL,             -- 가입일자
    grade VARCHAR(20) NOT NULL,          -- 등급
    points INTEGER NOT NULL              -- 포인트
);

-- 주문 테이블
CREATE TABLE orders (
    order_id VARCHAR(10) PRIMARY KEY,    -- 주문ID (예: O0001)
    customer_id VARCHAR(10) NOT NULL,    -- 고객ID
    product_id VARCHAR(10) NOT NULL,     -- 상품ID
    quantity INTEGER NOT NULL,           -- 수량
    order_date DATE NOT NULL,            -- 주문일자
    payment_method VARCHAR(20) NOT NULL, -- 결제수단
    payment_status VARCHAR(20) NOT NULL, -- 결제상태
    delivery_id VARCHAR(10) NOT NULL,    -- 배송ID
    total_amount INTEGER NOT NULL,       -- 총주문액
    used_points INTEGER NOT NULL,        -- 사용포인트
    discount_amount INTEGER NOT NULL,    -- 할인액
    shipping_fee INTEGER NOT NULL,       -- 배송비
    final_amount INTEGER NOT NULL,       -- 최종결제액
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id),
    FOREIGN KEY (delivery_id) REFERENCES deliveries(delivery_id)
);

-- 배송 테이블
CREATE TABLE deliveries (
    delivery_id VARCHAR(10) PRIMARY KEY, -- 배송ID (예: D0001)
    delivery_status VARCHAR(20) NOT NULL,-- 배송상태
    shipping_date DATE NOT NULL,         -- 출고일자
    expected_arrival DATE NOT NULL,      -- 도착예정일
    courier_company VARCHAR(50) NOT NULL,-- 배송사
    tracking_number VARCHAR(20) NOT NULL -- 송장번호
);
"""

In [4]:
####################################
# 2. 이커머스 샘플 데이터
####################################

# 상품 샘플 데이터
sample_products = """
INSERT INTO products VALUES('P001', '프리미엄 강아지 간식 세트', 'W02', '반려용품', 36, 32417, '2022-01-14');
INSERT INTO products VALUES('P002', '고양이 캣타워 XL', 'W02', '반려용품', 91, 47747, '2023-03-07');
INSERT INTO products VALUES('P003', '애완동물 자동 급식기', 'W02', '반려용품', 10, 22352, '2022-03-04');
INSERT INTO products VALUES('P004', '반려동물 이동가방', 'W03', '반려용품', 99, 91542, '2021-08-01');
INSERT INTO products VALUES('P005', '펫 전용 샴푸', 'W03', '반려용품', 49, 41276, '2020-12-26');
"""

# 고객 샘플 데이터
sample_customers = """
INSERT INTO customers VALUES('C001', '박예준', '2001-04-09', 'M', '017-612-4538', 'sbag@example.com', '2020-08-10', 'VIP', 5010);
INSERT INTO customers VALUES('C002', '김미숙', '1975-10-05', 'M', '052-368-8400', 'ohwang@example.org', '2024-01-23', 'VIP', 5771);
INSERT INTO customers VALUES('C003', '김예준', '2001-02-20', 'M', '061-192-5591', 'gimyejun@example.org', '2020-07-10', '일반', 4119);
INSERT INTO customers VALUES('C004', '곽하은', '1985-12-19', 'M', '053-366-1601', 'iyeji@example.org', '2021-06-17', 'VIP', 4641);
INSERT INTO customers VALUES('C005', '김도윤', '1991-02-01', 'M', '017-173-4341', 'gimsugja@example.net', '2023-09-17', '일반', 4434);
"""

# 배송 샘플 데이터 (orders 테이블보다 먼저 생성해야 FK 제약조건 충족)
sample_deliveries = """
INSERT INTO deliveries VALUES('D0001', '배송중', '2025-03-17', '2025-03-27', '우체국택배', '3885733726');
INSERT INTO deliveries VALUES('D0002', '배송중', '2025-03-07', '2025-03-25', 'CJ대한통운', '3845805571');
INSERT INTO deliveries VALUES('D0003', '배송중', '2025-03-20', '2025-03-27', '우체국택배', '9508993384');
INSERT INTO deliveries VALUES('D0004', '배송완료', '2025-01-24', '2025-03-25', '우체국택배', '2156991983');
INSERT INTO deliveries VALUES('D0005', '배송중', '2025-01-23', '2025-03-28', '한진택배', '8878855674');
"""

# 주문 샘플 데이터
sample_orders = """
INSERT INTO orders VALUES('O0001', 'C001', 'P001', 2, '2025-02-25', '카드', '보류', 'D0001', 144210, 640, 10848, 0, 132722);
INSERT INTO orders VALUES('O0002', 'C002', 'P002', 1, '2025-01-08', '휴대폰결제', '결제완료', 'D0002', 87735, 130, 13728, 0, 73877);
INSERT INTO orders VALUES('O0003', 'C003', 'P003', 1, '2025-01-26', '휴대폰결제', '결제완료', 'D0003', 22352, 510, 1030, 3000, 23812);
INSERT INTO orders VALUES('O0004', 'C001', 'P004', 1, '2025-02-04', '휴대폰결제', '취소', 'D0004', 77873, 841, 14649, 0, 62383);
INSERT INTO orders VALUES('O0005', 'C002', 'P005', 3, '2025-02-11', '계좌이체', '취소', 'D0005', 208998, 366, 2840, 0, 205792);
"""

In [5]:
# 전체 DDL과 샘플 데이터 합치기
ecommerce_complete_schema = ecommerce_schema + sample_deliveries + sample_products + sample_customers + sample_orders

sqlite DB 생성

In [6]:
from langchain_community.utilities import SQLDatabase
import tempfile
import sqlite3

db_path = './ecommerce.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

#초기화
cursor.executescript("""
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS deliveries;
""")

#DB 생성, 데이터 추가
cursor.executescript(ecommerce_schema)
cursor.executescript(sample_deliveries)
cursor.executescript(sample_products)
cursor.executescript(sample_customers)
cursor.executescript(sample_orders)

conn.commit()
conn.close()

db = SQLDatabase.from_uri(f'sqlite:///{db_path}')
print('사용 가능한 테이블 : ', db.get_usable_table_names())

사용 가능한 테이블 :  ['customers', 'deliveries', 'orders', 'products']


출력해보기

In [7]:
import ast

def parse_string_to_list(data_string: str) -> list:

    parsed_tuples = ast.literal_eval(data_string)

    list_of_lists = [list(item) for item in parsed_tuples]

    return list_of_lists


result = db.run("SELECT * FROM customers")
result = parse_string_to_list(result)
for r in result:
  print(r)

['C001', '박예준', '2001-04-09', 'M', '017-612-4538', 'sbag@example.com', '2020-08-10', 'VIP', 5010]
['C002', '김미숙', '1975-10-05', 'M', '052-368-8400', 'ohwang@example.org', '2024-01-23', 'VIP', 5771]
['C003', '김예준', '2001-02-20', 'M', '061-192-5591', 'gimyejun@example.org', '2020-07-10', '일반', 4119]
['C004', '곽하은', '1985-12-19', 'M', '053-366-1601', 'iyeji@example.org', '2021-06-17', 'VIP', 4641]
['C005', '김도윤', '1991-02-01', 'M', '017-173-4341', 'gimsugja@example.net', '2023-09-17', '일반', 4434]


#Agent 메인

sql 생성, 실행 함수

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

def generate_sql(prompt):
  system_prompt = """
  당신은 이커머스 DB SQL 전문가입니다. 사용자의 질문을 받아 해당 질문에 답하기 쉬운 SQL 쿼리를 생성해야 합니다.

  다음은 DB 스키마입니다:
  {schema}

  사용 가능한 테이블:
  {table_name}

  주어진 질문에 대한 정확한 SQL 쿼리를 생성하세요.
  SQL 쿼리 이외 다른 설명은 포함하지 마세요
  SQL 쿼리는 MySQL 문법을 사용해야 합니다.
  사용자 질문에 이전 대화 내용이 포함된 경우, 맥락에 따라 정확한 SQL 쿼리를 생성하세요.
  만약 질문이 SQL 쿼리로 해겨할 수 없는 경우 'SQL_QUERY_NOT_APPLICABLE'이라고 답변하세요.

  """
  user_prompt = "{question}"

  prompt_template = ChatPromptTemplate.from_messages([('system', system_prompt), ('user', user_prompt)])
  prompt = prompt_template.format_messages(
      schema = ecommerce_schema,
      table_name = db.get_usable_table_names(),
      question = prompt
  )

  response = llm.invoke(prompt)
  response = response.content

  if "```sql" in response:
    response = response.split("```sql")[1].split("```")[0].strip()
  elif "```" in response:
    response = response.split("```")[1].split("```")[0].strip()

  return response

In [21]:
def execute_sql(query):
  try:
    result = db.run(query)
    return result
  except Exception as e:
    return f'에러 발생 : {e}'

진행할 내용
1. 컨텍스트 없이 sql 변환만
2. sql 변환 후 llm 응답 받기
3. 컨텍스트 적용

In [49]:
####################################
# 4. 대화 컨텍스트 관리 클래스 정의
####################################

class ConversationContext:
    def __init__(self):
        self.history = []  # 대화 이력 저장 [(질문, 쿼리, 결과, 답변), ...]

    def add_interaction(self, question, query, result, answer):
        self.history.append((question, query, result, answer))

    def get_context_str(self):
        if not self.history:
            return ""

        context = "이전 대화 내용:\n"
        for i, (q, _, _, a) in enumerate(self.history[-3:]):  # 최근 3개 대화만 컨텍스트로 사용
            context += f"질문 {i+1}: {q}\n"
            context += f"답변 {i+1}: {a}\n"
        return context

# 대화 컨텍스트 인스턴스 생성
conversation_context = ConversationContext()

In [50]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [58]:
def process_question(question, use_context = True):

  #이전 컨텍스트 가져오기
  question_w_context = conversation_context.get_context_str() + "\n새로운 질문 : " + question
  print_with_decorations(question_w_context, 'question with context')

  #sql 구문 만들기
  sql_query = generate_sql(question_w_context)
  print_with_decorations(sql_query, 'sql_query')

  #1. SQL 쿼리로 해결 불가능한 질문일 경우
  if sql_query == "SQL_QUERY_NOT_APPLICABLE":
    #LLM에게 그냥 이 멍청이에게 친절하게 답변해다라는 prompt 작성 후 호출

    prompt_template = ChatPromptTemplate.from_messages(('user', """
    다음은 사용자의 질문입니다 :
    {question}

    이 질문은 이커머스 DB 쿼리로 해결할 수 없는 것으로 판단되었습니다.
    사용자에게 이 내용을 친절하게 제공하세요.
    그리고 DB를 통해 더 구체적인 정보를 얻으려면 어떻게 질문해야 할 지 안내해주세요.
    """))
    prompt = prompt_template.format_messages(question = question_w_context)
    response = llm.invoke(prompt)
    response = response.content

    conversation_context.add_interaction(question, 'SQL 적용 불가', '결과 없음', response)

    return response

  #2. SQL 쿼리로 해결 가능한 질문일 경우
  else:
    db_result = execute_sql(sql_query)

    prompt_template = ChatPromptTemplate.from_messages(('user', """
    다음은 사용자의 질문, SQL 쿼리, DB 쿼리 실행 결과입니다.
    질문 : {question}
    SQL 쿼리 : {sql_query}
    결과 : {db_result}

    이 정보를 바탕으로 사용자 질문에 답해주세요.
    SQL 쿼리, 기술적인 부분은 제외하고 일반인이 이해할 수 있는 수준으로 친절히 답변하세요.
    질문에 대한 답이 결과에 없다면 알 수 없다고 안내하세요.

    마크다운 형식은 사용하지 마세요.
    """))
    prompt = prompt_template.format_messages(
        question = question_w_context,
        sql_query = sql_query,
        db_result = db_result)
    response = llm.invoke(prompt)
    response = response.content

    print_with_decorations(db_result, 'sql 실행 결과')

    conversation_context.add_interaction(question, sql_query, db_result, response)

    return response

In [59]:
response1 = process_question('주문 시 휴대폰결제를 사용한 사람들의 아이디')


=========================question with context=========================
이전 대화 내용:
질문 1: 주문 시 휴대폰결제를 사용한 사람들의 아이디
답변 1: 주문 시 휴대폰 결제를 사용한 고객들의 아이디는 C002, C003, C001입니다. 이 고객들은 모두 휴대폰 결제를 선택하여 주문을 하신 분들입니다.
질문 2: 이번엔 카드나 계좌이체를 사용한 사람들
답변 2: 이번에 카드나 계좌이체를 사용한 고객들의 아이디는 C001과 C002입니다. 이 두 고객은 주문 시 카드나 계좌이체를 선택하여 결제를 하신 분들입니다.

새로운 질문 : 주문 시 휴대폰결제를 사용한 사람들의 아이디


=========================sql_query=========================
SELECT DISTINCT customer_id 
FROM orders 
WHERE payment_method = '휴대폰결제';


=========================sql 실행 결과=========================
[('C002',), ('C003',), ('C001',)]



In [60]:
response2 = process_question('이번엔 카드나 계좌이체를 사용한 사람들')


=========================question with context=========================
이전 대화 내용:
질문 1: 주문 시 휴대폰결제를 사용한 사람들의 아이디
답변 1: 주문 시 휴대폰 결제를 사용한 고객들의 아이디는 C002, C003, C001입니다. 이 고객들은 모두 휴대폰 결제를 선택하여 주문을 하신 분들입니다.
질문 2: 이번엔 카드나 계좌이체를 사용한 사람들
답변 2: 이번에 카드나 계좌이체를 사용한 고객들의 아이디는 C001과 C002입니다. 이 두 고객은 주문 시 카드나 계좌이체를 선택하여 결제를 하신 분들입니다.
질문 3: 주문 시 휴대폰결제를 사용한 사람들의 아이디
답변 3: 주문 시 휴대폰 결제를 사용한 고객들의 아이디는 C002, C003, C001입니다. 이 고객들은 모두 휴대폰 결제를 선택하여 주문을 하신 분들입니다.

새로운 질문 : 이번엔 카드나 계좌이체를 사용한 사람들


=========================sql_query=========================
SELECT DISTINCT customer_id 
FROM orders 
WHERE payment_method IN ('카드', '계좌이체');


=========================sql 실행 결과=========================
[('C001',), ('C002',)]



In [62]:
response3 = process_question('VIP 고객 수')


=========================question with context=========================
이전 대화 내용:
질문 1: 이번엔 카드나 계좌이체를 사용한 사람들
답변 1: 이번에 카드나 계좌이체를 사용한 고객들의 아이디는 C001과 C002입니다. 이 두 고객은 주문 시 카드나 계좌이체를 선택하여 결제를 하신 분들입니다.
질문 2: 주문 시 휴대폰결제를 사용한 사람들의 아이디
답변 2: 주문 시 휴대폰 결제를 사용한 고객들의 아이디는 C002, C003, C001입니다. 이 고객들은 모두 휴대폰 결제를 선택하여 주문을 하신 분들입니다.
질문 3: 이번엔 카드나 계좌이체를 사용한 사람들
답변 3: 이번에 카드나 계좌이체를 사용한 고객들의 아이디는 C001과 C002입니다. 이 두 고객은 주문 시 카드나 계좌이체를 선택하여 결제를 하신 분들입니다.

새로운 질문 : VIP 고객 수


=========================sql_query=========================
SELECT COUNT(*) AS vip_customer_count
FROM customers
WHERE grade = 'VIP';


=========================sql 실행 결과=========================
[(3,)]



In [64]:
response4 = process_question('이번엔 아닌 고객 수')
response4


=========================question with context=========================
이전 대화 내용:
질문 1: 주문 시 휴대폰결제를 사용한 사람들의 아이디
답변 1: 주문 시 휴대폰 결제를 사용한 고객들의 아이디는 C002, C003, C001입니다. 이 고객들은 모두 휴대폰 결제를 선택하여 주문을 하신 분들입니다.
질문 2: 이번엔 카드나 계좌이체를 사용한 사람들
답변 2: 이번에 카드나 계좌이체를 사용한 고객들의 아이디는 C001과 C002입니다. 이 두 고객은 주문 시 카드나 계좌이체를 선택하여 결제를 하신 분들입니다.
질문 3: VIP 고객 수
답변 3: 이번에 VIP 고객 수는 3명입니다. 이 고객들은 특별한 혜택을 받는 등 VIP 등급으로 분류된 고객들입니다. 추가로 궁금한 점이 있으시면 언제든지 질문해 주세요!

새로운 질문 : 이번엔 아닌 고객 수


=========================sql_query=========================
SQL_QUERY_NOT_APPLICABLE



'안녕하세요! 질문해 주셔서 감사합니다. "이번엔 아닌 고객 수"라는 질문은 이커머스 데이터베이스에서 직접적으로 쿼리로 해결하기 어려운 부분입니다. 이는 "아닌 고객"의 정의가 명확하지 않기 때문입니다. 예를 들어, 어떤 기준으로 고객을 \'아닌\' 고객으로 분류할 것인지에 대한 정보가 필요합니다.\n\n더 구체적인 정보를 얻으시려면 다음과 같은 질문을 해보시는 것이 좋습니다:\n\n1. "주문을 하지 않은 고객 수는 몇 명인가요?"\n2. "특정 결제 방법을 사용하지 않은 고객 수는 얼마인가요?"\n3. "VIP 고객이 아닌 고객 수는 얼마인가요?"\n\n이와 같은 질문을 통해 보다 명확한 데이터를 요청하실 수 있습니다. 추가로 궁금한 점이 있으시면 언제든지 말씀해 주세요!'

In [61]:
def print_with_decorations(text, title):
  top_decoration = '='*25 + title + '='*25

  print()
  print(top_decoration)
  print(text)
  print('='*len(top_decoration))
  print()

In [66]:
import gradio as gr

In [67]:
# ChatInterface를 사용해 간단하고 세련된 챗봇 화면 구성
demo = gr.ChatInterface(
    fn=process_question,
    title="🛒 이커머스 DB AI 어시스턴트",
    description="자연어로 데이터베이스를 조회해 보세요! (예: '주문 시 휴대폰결제를 사용한 사람들의 아이디는?', '이번엔 카드나 계좌이체를 사용한 사람들은?')",
    examples=["주문 시 휴대폰결제를 사용한 사람들의 아이디는?", "VIP 고객 수는 몇 명이야?"],
    theme="soft"
)

if __name__ == "__main__":
    demo.launch(share=False) # 외부 공유 링크가 필요하면 share=True로 변경

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [68]:
conversation_context.history

[('주문 시 휴대폰결제를 사용한 사람들의 아이디',
  "SELECT DISTINCT customer_id \nFROM orders \nWHERE payment_method = '휴대폰결제';",
  "[('C002',), ('C003',), ('C001',)]",
  '주문 시 휴대폰 결제를 사용한 고객들의 아이디는 C002, C003, C001입니다. 이 고객들은 모두 휴대폰 결제를 선택하여 주문을 하신 분들입니다.'),
 ('이번엔 카드나 계좌이체를 사용한 사람들',
  "SELECT DISTINCT customer_id \nFROM orders \nWHERE payment_method IN ('카드', '계좌이체');",
  "[('C001',), ('C002',)]",
  '이번에 카드나 계좌이체를 사용한 고객들의 아이디는 C001과 C002입니다. 이 두 고객은 주문 시 카드나 계좌이체를 선택하여 결제를 하신 분들입니다.'),
 ('주문 시 휴대폰결제를 사용한 사람들의 아이디',
  "SELECT DISTINCT customer_id \nFROM orders \nWHERE payment_method = '휴대폰결제';",
  "[('C002',), ('C003',), ('C001',)]",
  '주문 시 휴대폰 결제를 사용한 고객들의 아이디는 C002, C003, C001입니다. 이 고객들은 모두 휴대폰 결제를 선택하여 주문을 하신 분들입니다.'),
 ('이번엔 카드나 계좌이체를 사용한 사람들',
  "SELECT DISTINCT customer_id \nFROM orders \nWHERE payment_method IN ('카드', '계좌이체');",
  "[('C001',), ('C002',)]",
  '이번에 카드나 계좌이체를 사용한 고객들의 아이디는 C001과 C002입니다. 이 두 고객은 주문 시 카드나 계좌이체를 선택하여 결제를 하신 분들입니다.'),
 ('VIP 고객 수',
  "SELECT COUNT(*) AS vip_customer